# 02 - Precipitation Deterministic Forecasts

## Learning goals of this module
- Learn how to do a basic analysis, comparing observed precipitation with forecast precipitation from GDPS
- Learn how to calculate and visualize simple error metrics

## Assumptions
- We assume you are familiar with the concept of a __forecast__, and a __lead_time__. 

### Reference to data products

- [Global Deterministic Prediction System (GDPS)](https://open.canada.ca/data/en/dataset/c041e79a-914a-5a4e-a485-9cbc506195df)


## Run imports and set-up logging

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

from dotenv import load_dotenv

from veriflow import run_pipeline
from veriflow.constants import VERSION

# add project root (parent of notebook folder) to path
sys.path.append(str(Path("..").resolve()))


from tree_plots import (
    plot_score_vs_lead_time,
    forecast_timeseries_plot,
    get_pair_dataset,
    reanalysis_timeseries_plot
)

# Reload automatically
%load_ext autoreload
%autoreload 2

load_dotenv(dotenv_path="tutorial.env", override=True)

base_config = Path("config")
base_config.exists()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logging.info(f"Running Veriflow version {VERSION}")

2026-11-04 14:58:06,428 - INFO - Running Veriflow version 0.4.1


## Inspecting the _veriflow_ pipeline configuration
1. Open the config file in the "config" directory. The name of the file is identical to the name of the notebook.
2. Inspect each of the sections to gain an understanding of what this configuration is about.

## Running the _veriflow_ pipeline

In [3]:
dt = run_pipeline((base_config / "02_elbow_deterministic_forecast.yaml", "yaml"))

2026-11-04 14:58:12,945 - INFO - Successfully initialized the configuration. 
	 verification_period_start = 2026-05-15 00:00:00 
	 verification_period_end = 2026-06-01 00:00:00
2026-11-04 14:58:12,948 - INFO - Starting dataset fetch (source_id=observed) from FewsWebservice.
2026-11-04 14:58:13,413 - INFO - Download successful from URL: https://veriflow-open.fews.deltares.nl/FewsWebServices/rest/fewspiservice/v1/timeseries?locationIds=3031092&locationIds=3050778&locationIds=MSC-005&locationIds=05BL813&locationIds=05BJ804&locationIds=05BL809&locationIds=FIRES-B4&locationIds=05BJ805&locationIds=05BL812&locationIds=FIRES-B5&locationIds=05BJ806&locationIds=05BH803&locationIds=05BF825&locationIds=05BH802&locationIds=05BL810&locationIds=05BF827&parameterIds=PC.obs&moduleInstanceIds=ImportMeteoStations&startTime=2026-05-15T00%3A00%3A00Z&endTime=2026-06-01T00%3A00%3A00Z&timeSeriesType=EXTERNAL_HISTORICAL&documentFormat=PI_NETCDF&omitEmptyTimeSeries=True
2026-11-04 14:58:13,416 - INFO - Starting

## Evaluating the results in the _veriflow_ output `DataTree`

Verification metrics and results can contain a level of abstraction. Although these abstractions can reveal important information about forecast quality, a basic "eyeball verification" is often the best and intuitive way to start your verification exercise. You'll likely find strengths and weaknesses in your forecasts early on, without directly diving into levels of abstraction. In addition, a solid visual inspection may help you later on in understanding or explaining the more abstract results.

### 1 - Visual inspection of observed and forecast data
A good starting point for "eyeball" verification is simple: just looking at your observations and forecasts in a visual way. Use the interactive elements in the plots below to zoom, pan and compare the results of our 3 NWP products.

In [8]:
ds = dt.veriflow.get_aligned_input(dt.veriflow.verification_pairs[0])["observations"].to_dataset()
ds

<xarray.Dataset> Size: 13kB
Dimensions:                  (station: 16, forecast_reference_time: 17,
                              lead_time: 7)
Coordinates:
  * station                  (station) <U8 512B '3031092' ... '05BF827'
    station_name             (station) |S255 4kB b'Calgary International' ......
    lat                      (station) float64 128B 51.12 51.08 ... 50.65 50.87
    lon                      (station) float64 128B -114.0 -115.1 ... -114.3
    y                        (station) float64 128B 51.12 51.08 ... 50.65 50.87
    x                        (station) float64 128B -114.0 -115.1 ... -114.3
    z                        (station) float64 128B 1.099e+03 1.321e+03 ... nan
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 136B 20...
  * lead_time                (lead_time) timedelta64[ns] 56B 1 days ... 7 days
Data variables:
    PC                       (station, forecast_reference_time, lead_time) float32 8kB ...

In [10]:
stations = ds.coords["station"].values
lead_times = ds.coords["lead_time"].values

lead_times_hours = [str(lt.astype("timedelta64[h]")) for lt in lead_times]
print(f"Stations: {stations}")
print(f"GDPS Lead times (hours): {lead_times_hours}")

Stations: ['3031092' '3050778' 'MSC-005' '05BL813' '05BJ804' '05BL809' 'FIRES-B4'
 '05BJ805' '05BL812' 'FIRES-B5' '05BJ806' '05BH803' '05BF825' '05BH802'
 '05BL810' '05BF827']
GDPS Lead times (hours): ['24 hours', '48 hours', '72 hours', '96 hours', '120 hours', '144 hours', '168 hours']


In [5]:
forecast_timeseries_plot(dt, station=stations[1])

### 2 - Looking into the Mean Error per lead time



In [7]:
# Plot the first 5 stations to avoid overcrowding the plot
plot_score_vs_lead_time(dt, score_name="continuous_scores", score_var="mean_error", stations=stations[:5])